In [ ]:
#hibridinio modelio konfiguracija
import os, json, random, warnings, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

#kad rezultatai butu vienodi
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    print(f"TensorFlow: {tf.__version__}  (seed={SEED})")
except ImportError:
    raise RuntimeError("Reikalinga tensorflow: pip install tensorflow")

#datos
TRAIN_START  = pd.Timestamp("2020-04-01")
EFF_START    = pd.Timestamp("2021-03-31")   #po t-364 NaN
BLIND_START  = pd.Timestamp("2026-01-01")
BLIND_END    = pd.Timestamp("2026-03-31")

#visus modelius su parametrais ir kitais rezultatai laikyti viename aplanke (PARENT)
PARENT       = r"C:\Users\..."
CSV_PATH     = os.path.join(PARENT, "duomenys.csv")
LOG_PATH     = "Hibridinis_Zurnalas.csv"
OOF_CACHE    = "Hibridinis_OOF_cache.json"
META_PARAMS  = "Hibridinis_meta_params.json"

#baziniu modeliu artefaktai
XGB_PARAMS   = os.path.join(PARENT, "XGBoost_best_params.json")
SAR_PARAMS   = os.path.join(PARENT, "SARIMAX_best_params.json")
GRU_PARAMS   = os.path.join(PARENT, "GRU_best_params.json")
GRU_SCALER   = os.path.join(PARENT, "GRU_scaler.json")

#baziniu modeliu zurnalai
XGB_LOG      = os.path.join(PARENT, "XGBoost_Zurnalas.csv")
SAR_LOG      = os.path.join(PARENT, "SARIMAX_Zurnalas.csv")
GRU_LOG      = os.path.join(PARENT, "GRU_Zurnalas.csv")

#paskutine diena su faktais
_raw_csv = pd.read_csv(CSV_PATH)
_raw_csv.columns = ['Data'] + list(_raw_csv.columns[1:])
_raw_csv['Data'] = pd.to_datetime(_raw_csv['Data'], dayfirst=True)
_col_b = _raw_csv.iloc[:, 1]
_mask = _col_b.notna() & (_col_b != '') & (_col_b != 0)
TRAIN_END = pd.Timestamp(_raw_csv.loc[_mask, 'Data'].max())
#apsauga: train_end negali but po blind_start (no leakage)
_dyn_end = TRAIN_END
if TRAIN_END >= BLIND_START:
    TRAIN_END = BLIND_START - pd.Timedelta(days=1)
    print(f"[!] CSV iki {_dyn_end.date()}, capping ant {TRAIN_END.date()} (no leakage)")
del _raw_csv, _col_b, _mask
print(f"Mokymo periodas: {TRAIN_START.date()} -> {TRAIN_END.date()}")

#lag zingsniai ir cv parametrai
LAGS         = (1, 2, 3, 7, 14, 28, 364)
K_FOLDS      = 5            #timeseriessplit klosciu
CV_GAP       = 14           #atstumas tarp train/test (ilgiausias rolling=14)
ALPHA_CI     = 0.10         #90% CI
BOOTSTRAP_N  = 200          #bootstrap CI iteraciju

#metrikos
def mae(y, yhat):  return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat))**2)))
def mape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    mask = y != 0
    return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100) if mask.any() else np.nan
def smape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    denom = (np.abs(y) + np.abs(yhat)) / 2.0
    mask = denom != 0
    return float(np.mean(np.abs(y[mask] - yhat[mask]) / denom[mask]) * 100) if mask.any() else np.nan
def mase(y, yhat, y_hist, m=7):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    y_hist = np.asarray(y_hist, dtype=float)
    denom = float(np.mean(np.abs(y_hist[m:] - y_hist[:-m])))
    return float(np.mean(np.abs(y - yhat)) / denom) if denom > 0 else np.nan
def picp(y, lo, hi): return float(np.mean((y >= lo) & (y <= hi)) * 100)
def pinaw(lo, hi, y_range): return float(np.mean(hi - lo) / y_range) if y_range > 0 else np.nan
def winkler(y, lo, hi, alpha=ALPHA_CI):
    total = 0.0
    for i in range(len(y)):
        w = hi[i] - lo[i]
        if   y[i] < lo[i]: total += w + (2.0/alpha) * (lo[i] - y[i])
        elif y[i] > hi[i]: total += w + (2.0/alpha) * (y[i] - hi[i])
        else:              total += w
    return float(total / len(y))

def savaites_nr_metuose(data: pd.Timestamp) -> int:
    jan1 = pd.Timestamp(year=data.year, month=1, day=1)
    pirmas_sekm = jan1 + pd.Timedelta(days=(6 - jan1.dayofweek) % 7)
    if data <= pirmas_sekm:
        return 1
    return 2 + (data - (pirmas_sekm + pd.Timedelta(days=1))).days // 7

print(f"Konfig: mokom {TRAIN_START.date()} -> {TRAIN_END.date()}, blind {BLIND_START.date()} -> {BLIND_END.date()}")
print(f"K-fold OOF: {K_FOLDS} klosciu, gap={CV_GAP} d., bootstrap N={BOOTSTRAP_N}")

#protingas keso nuvertinimas
#oof cache yra brangus (1-3h gru), atnaujinam tik kai duomenys pasikeite
_needs_refresh = False
if os.path.exists(OOF_CACHE):
    try:
        with open(OOF_CACHE, "r", encoding="utf-8") as _f:
            _cache_d = json.load(_f)
        _saved_end = _cache_d.get("_train_end", None)
        if _saved_end != str(TRAIN_END.date()):
            _needs_refresh = True
            print(f"[!] Duomenys pasikeite ({_saved_end} -> {TRAIN_END.date()}); cache atnaujinamas.")
        else:
            cache_dates_max = pd.Timestamp(max(_cache_d.get("dates", ["2000-01-01"])))
            if cache_dates_max > TRAIN_END:
                _needs_refresh = True
                print(f"[!] Cache turi 2026 leakage ({cache_dates_max.date()} > {TRAIN_END.date()}); atnaujinamas.")
    except Exception as _e:
        _needs_refresh = True
        print(f"[!] Cache klaida: {_e}; atnaujinamas.")

if _needs_refresh:
    for _cf in (OOF_CACHE, META_PARAMS):
        if os.path.exists(_cf):
            os.remove(_cf)
            print(f"    Istrintas: {_cf}")
elif os.path.exists(OOF_CACHE) and os.path.exists(META_PARAMS):
    print(f"[OK] Cache atitinka TRAIN_END={TRAIN_END.date()}; bus naudojamas.")
else:
    print(f"[i] Cache nera; OOF + meta apmokoma is naujo.")


In [ ]:
#duomenys ir pozymiai (toks pat rinkinys kaip xgboost - kad meta-modelis matytu ta pati konteksta)

#nuskaitymas
df_raw = pd.read_csv(CSV_PATH)
df_raw["Data"] = pd.to_datetime(df_raw["Data"], dayfirst=True, errors="coerce")
df_raw = df_raw.dropna(subset=["Data"]).sort_values("Data").set_index("Data")

paskutine_CSV = df_raw.index.max()
pilnas_indeksas = pd.date_range(TRAIN_START, paskutine_CSV, freq="D")
df = df_raw.reindex(pilnas_indeksas)
df.index.name = "Data"
print(f"Duomenys: {TRAIN_START.date()} -> {paskutine_CSV.date()} ({len(df)} d., truksta: {int(df['Skambuciai'].isna().sum())})")

#trukstamu uzpildymas
df["Skambuciai"] = df["Skambuciai"].interpolate(method="linear", limit=3, limit_area="inside")
for dt in df.index[df["Skambuciai"].isna()]:
    kand = []
    for y in range(TRAIN_START.year, dt.year):
        past_date = dt - pd.DateOffset(years=(dt.year - y))
        if past_date in df.index and not pd.isna(df.at[past_date, "Skambuciai"]):
            kand.append(float(df.at[past_date, "Skambuciai"]))
    if kand:
        df.at[dt, "Skambuciai"] = float(np.median(kand))
df["Skambuciai"] = df["Skambuciai"].fillna(df["Skambuciai"].shift(1).rolling(7, min_periods=1).mean())

#LT sventes
LT_SVENTES = pd.to_datetime([
    "2020-01-01","2021-01-01","2022-01-01","2023-01-01","2024-01-01","2025-01-01","2026-01-01",
    "2020-02-16","2021-02-16","2022-02-16","2023-02-16","2024-02-16","2025-02-16","2026-02-16",
    "2020-03-11","2021-03-11","2022-03-11","2023-03-11","2024-03-11","2025-03-11","2026-03-11",
    "2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05",
    "2020-04-13","2021-04-05","2022-04-18","2023-04-10","2024-04-01","2025-04-21","2026-04-06",
    "2020-05-01","2021-05-01","2022-05-01","2023-05-01","2024-05-01","2025-05-01","2026-05-01",
    "2020-06-24","2021-06-24","2022-06-24","2023-06-24","2024-06-24","2025-06-24","2026-06-24",
    "2020-07-06","2021-07-06","2022-07-06","2023-07-06","2024-07-06","2025-07-06","2026-07-06",
    "2020-08-15","2021-08-15","2022-08-15","2023-08-15","2024-08-15","2025-08-15","2026-08-15",
    "2020-11-01","2021-11-01","2022-11-01","2023-11-01","2024-11-01","2025-11-01","2026-11-01",
    "2020-11-02","2021-11-02","2022-11-02","2023-11-02","2024-11-02","2025-11-02","2026-11-02",
    "2020-12-24","2021-12-24","2022-12-24","2023-12-24","2024-12-24","2025-12-24","2026-12-24",
    "2020-12-25","2021-12-25","2022-12-25","2023-12-25","2024-12-25","2025-12-25","2026-12-25",
    "2020-12-26","2021-12-26","2022-12-26","2023-12-26","2024-12-26","2025-12-26","2026-12-26",
])
VELYKOS  = pd.to_datetime(["2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05"])
SEKMINES = pd.to_datetime(["2020-05-31","2021-05-23","2022-06-05","2023-05-28","2024-05-19","2025-06-08","2026-05-24"])
SVENT_SET = set(LT_SVENTES); VEL_SET = set(VELYKOS); SEK_SET = set(SEKMINES)

#pozymiai
def sukurti_pozymius(series: pd.DataFrame) -> pd.DataFrame:
    d = series.copy()
    idx = pd.Series(d.index, index=d.index)
    dow = d.index.dayofweek
    mon = d.index.month
    d["week_iso"]  = d.index.isocalendar().week.astype(int)
    d["day"]       = d.index.day
    d["quarter"]   = d.index.quarter
    d["dayofyear"] = d.index.dayofyear
    d["is_weekend"] = (dow >= 5).astype(int)
    d["is_holiday"]         = idx.isin(SVENT_SET).astype(int).values
    d["day_before_holiday"] = (idx + pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["day_after_holiday"]  = (idx - pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["is_easter"]          = idx.isin(VEL_SET).astype(int).values
    d["is_pentecost"]       = idx.isin(SEK_SET).astype(int).values
    d["dow_sin"] = np.sin(2*np.pi*dow/7); d["dow_cos"] = np.cos(2*np.pi*dow/7)
    d["moy_sin"] = np.sin(2*np.pi*mon/12); d["moy_cos"] = np.cos(2*np.pi*mon/12)
    for lag in LAGS:
        d[f"lag_{lag}"] = d["Skambuciai"].shift(lag)
    s1 = d["Skambuciai"].shift(1)
    d["roll_mean_7"]  = s1.rolling(7).mean()
    d["roll_mean_14"] = s1.rolling(14).mean()
    d["roll_mean_30"] = s1.rolling(30).mean()
    d["roll_std_7"]   = s1.rolling(7).std()
    d["roll_med_7"]   = s1.rolling(7).median()
    d["roll_min_7"]   = s1.rolling(7).min()
    d["roll_max_7"]   = s1.rolling(7).max()
    return d

df_feat = sukurti_pozymius(df[["Skambuciai"]])
FEATURE_COLS = [c for c in df_feat.columns if c != "Skambuciai"]
print(f"Pozymiu skaicius: {len(FEATURE_COLS)}")
print(f"Pirma diena su visais lag: {df_feat.dropna(subset=[f'lag_{l}' for l in LAGS]).index.min().date()}")

#efektyvus mokymo rinkinys
D_eff = df_feat[(df_feat.index >= EFF_START) & (df_feat.index <= TRAIN_END)] \
           .dropna(subset=[f"lag_{l}" for l in LAGS])
print(f"D_eff: {D_eff.index.min().date()} -> {D_eff.index.max().date()} ({len(D_eff)} d.)")


In [ ]:
#baziniu modeliu artefaktu nuskaitymas
#hibridinis nepakartoja hpo - naudoja jau optimizuotus parametrus

required_files = {
    "XGBoost params":  XGB_PARAMS,
    "SARIMAX params":  SAR_PARAMS,
    "GRU params":      GRU_PARAMS,
    "GRU scaler":      GRU_SCALER,
}
missing = [name for name, path in required_files.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        f"Truksta: {missing}\n"
        "Pirma paleisk visus 3 bazinius notebookus."
    )

#xgboost
with open(XGB_PARAMS, "r", encoding="utf-8") as f:
    xgb_params = json.load(f)
xgb_params.pop("_train_end", None)
print(f"XGBoost params: {len(xgb_params)} parametru")
print(f"   max_depth={xgb_params.get('max_depth')}, n_estimators={xgb_params.get('n_estimators')}, learning_rate={xgb_params.get('learning_rate', 0):.4f}")

#sarimax
with open(SAR_PARAMS, "r", encoding="utf-8") as f:
    sar_params = json.load(f)
sar_params.pop("_train_end", None)
SAR_ORDER          = tuple(sar_params["order"])
SAR_SEASONAL_ORDER = tuple(sar_params["seasonal_order"])
print(f"SARIMAX params: order={SAR_ORDER}, seasonal_order={SAR_SEASONAL_ORDER}")

#gru
with open(GRU_PARAMS, "r", encoding="utf-8") as f:
    gru_params = json.load(f)
gru_params.pop("_train_end", None)
#issvalom keras tuner tarnybinius raktus
for _k in list(gru_params.keys()):
    if _k.startswith("tuner/"):
        gru_params.pop(_k, None)
print(f"GRU params: lookback={gru_params.get('lookback')}, n_layers={gru_params.get('n_layers')}, units={gru_params.get('units')}, loss={gru_params.get('loss')}")

with open(GRU_SCALER, "r", encoding="utf-8") as f:
    gru_scaler = json.load(f)
print(f"GRU scaler: y_min={gru_scaler['y_min']:.0f}, y_max={gru_scaler['y_max']:.0f}, {len(gru_scaler['feature_cols'])} pozymiai")

print("\nVisi 4 baziniu modeliu artefaktai pakrauti.")


In [ ]:
#oof (out-of-fold) prognoziu generavimas
#brangiausias zingsnis - 5 klostys x 3 modeliai. pirmas paleidimas ~1-3h. po to viskas keso.

from sklearn.model_selection import TimeSeriesSplit

if os.path.exists(OOF_CACHE):
    print("OOF cache atitinka TRAIN_END; nuskaitomi issaugotos prognozes...")
    with open(OOF_CACHE, "r", encoding="utf-8") as f:
        oof_data = json.load(f)
    oof_dates    = pd.DatetimeIndex(oof_data["dates"])
    oof_y_true   = np.array(oof_data["y_true"], dtype=float)
    oof_xgb      = np.array(oof_data["xgb"], dtype=float)
    oof_sar      = np.array(oof_data["sarimax"], dtype=float)
    oof_gru      = np.array(oof_data["gru"], dtype=float)
    if oof_dates.max() > TRAIN_END:
        raise ValueError(
            f"OOF leakage! Cache turi {oof_dates.max().date()}, bet TRAIN_END={TRAIN_END.date()}. "
            f"Istrink {OOF_CACHE} ir paleisk is naujo."
        )
    print(f"   OOF dydis: {len(oof_dates)} d. ({oof_dates.min().date()} -> {oof_dates.max().date()})")
else:
    print("Generuojam OOF prognozes (TimeSeriesSplit K=5)...")
    print(f"   Numatomas laikas: ~1-3h (priklauso nuo gru greicio)\n")

    #oof generavimo funkcijos kiekvienam baziniam modeliui
    def _oof_xgb(D_train, D_test, params):
        import xgboost as xgb
        X_tr = D_train[FEATURE_COLS].values
        y_tr = D_train["Skambuciai"].values.astype(float)
        y_tr_log = np.log1p(y_tr)  #xgboost notebook'e log1p

        bp = {k: v for k, v in params.items() if k != "_train_end"}
        m = xgb.XGBRegressor(**bp)
        m.fit(X_tr, y_tr_log)

        X_te = D_test[FEATURE_COLS].fillna(0.0).values
        y_pred_log = m.predict(X_te)
        y_pred = np.expm1(y_pred_log)
        return np.maximum(0.0, y_pred)

    def _oof_sarimax(D_train, D_test, order, seasonal_order):
        from statsmodels.tsa.statespace.sarimax import SARIMAX

        EXOG_COLS = ["is_weekend","is_holiday","day_before_holiday","day_after_holiday",
                     "is_easter","is_pentecost"]
        for c in EXOG_COLS:
            if c not in D_train.columns:
                raise KeyError(f"Truksta egz: {c}")

        y_tr_log = np.log1p(D_train["Skambuciai"].values.astype(float))
        X_tr = D_train[EXOG_COLS].values
        X_te = D_test[EXOG_COLS].values

        try:
            m = SARIMAX(endog=y_tr_log, exog=X_tr,
                        order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
            rez = m.fit(disp=False, maxiter=300)
            fc = rez.get_forecast(steps=len(D_test), exog=X_te)
            y_pred_log = fc.predicted_mean
            y_pred = np.expm1(y_pred_log)
            return np.maximum(0.0, y_pred.values if hasattr(y_pred, "values") else y_pred)
        except Exception as e:
            print(f"     SARIMAX nepavyko ({e}); fallback naive-7")
            fallback = D_train["Skambuciai"].tail(7).median()
            return np.full(len(D_test), float(fallback))

    def _oof_gru(D_train, D_test, params):
        from tensorflow.keras import layers, models, optimizers, losses, callbacks
        tf.random.set_seed(SEED)

        LOOKBACK = int(params["lookback"])

        #scaler tik ant D_train (no leakage)
        feat_min = D_train[FEATURE_COLS].min().to_dict()
        feat_max = D_train[FEATURE_COLS].max().to_dict()
        y_min = float(D_train["Skambuciai"].min())
        y_max = float(D_train["Skambuciai"].max())

        def _scale_X(X_arr, fc):
            lo = np.array([feat_min[c] for c in fc], dtype=float)
            hi = np.array([feat_max[c] for c in fc], dtype=float)
            rng = np.where(hi - lo == 0, 1.0, hi - lo)
            return (X_arr - lo) / rng

        def _scale_y(yv): return (yv - y_min) / max(1e-9, y_max - y_min)
        def _inv_y(ys):   return ys * (y_max - y_min) + y_min

        def _make_seq(d_local):
            X_raw = d_local[FEATURE_COLS].values.astype(float)
            y_raw = d_local["Skambuciai"].values.astype(float)
            X_n = _scale_X(X_raw, FEATURE_COLS)
            y_n = _scale_y(y_raw)
            n = len(d_local)
            if n < LOOKBACK:
                return (np.empty((0, LOOKBACK, len(FEATURE_COLS))),
                        np.empty((0,)), d_local.index[:0])
            X_seq = np.stack([X_n[i - LOOKBACK + 1: i + 1] for i in range(LOOKBACK - 1, n)])
            y_seq = y_n[LOOKBACK - 1: n]
            idx   = d_local.index[LOOKBACK - 1: n]
            return X_seq, y_seq, idx

        X_tr_s, y_tr_s, idx_tr = _make_seq(D_train)
        if len(X_tr_s) < 50:
            print(f"     GRU: per maza aibe ({len(X_tr_s)}); fallback")
            fallback = D_train["Skambuciai"].tail(7).median()
            return np.full(len(D_test), float(fallback))

        bp = {k: v for k, v in params.items() if k != "_train_end" and not k.startswith("tuner/")}
        n_layers = int(bp.get("n_layers", 1))
        units = int(bp.get("units", 64))
        dropout = float(bp.get("dropout", 0.2))
        rec_dropout = float(bp.get("rec_dropout", 0.1))
        lr = float(bp.get("learning_rate", 0.005))
        loss_name = bp.get("loss", "huber")
        bs = int(bp.get("batch_size", 16))

        inp = layers.Input(shape=(LOOKBACK, len(FEATURE_COLS)))
        x = inp
        for i in range(n_layers):
            x = layers.GRU(
                units, return_sequences=(i < n_layers - 1),
                dropout=dropout, recurrent_dropout=rec_dropout,
                activation="tanh", recurrent_activation="sigmoid"
            )(x)
        out = layers.Dense(1, activation="linear")(x)
        m = models.Model(inp, out)
        loss_fn = losses.Huber(delta=1.0) if loss_name == "huber" else losses.MeanSquaredError()
        m.compile(optimizer=optimizers.Adam(learning_rate=lr, clipnorm=1.0), loss=loss_fn)

        split = int(len(X_tr_s) * 0.85)
        es = callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss")
        m.fit(X_tr_s[:split], y_tr_s[:split],
              validation_data=(X_tr_s[split:], y_tr_s[split:]),
              epochs=50, batch_size=bs, verbose=0, callbacks=[es])

        #prognoze
        y_pred = []
        for dt in D_test.index:
            window_end_pos = D_test.index.get_loc(dt)
            full_combined = pd.concat([D_train, D_test.iloc[:window_end_pos]])
            window = full_combined.tail(LOOKBACK)
            if len(window) < LOOKBACK:
                y_pred.append(float(D_train["Skambuciai"].tail(7).median()))
                continue
            X_w = _scale_X(window[FEATURE_COLS].values.astype(float), FEATURE_COLS)
            X_w = X_w.reshape(1, LOOKBACK, len(FEATURE_COLS))
            y_p_s = float(m.predict(X_w, verbose=0).flatten()[0])
            y_p = _inv_y(np.array([y_p_s]))[0]
            y_pred.append(max(0.0, float(y_p)))

        return np.array(y_pred)


    #k-fold valdiklis
    tscv = TimeSeriesSplit(n_splits=K_FOLDS, gap=CV_GAP)
    D_eff_arr = D_eff.copy()
    n_total = len(D_eff_arr)

    oof_xgb = np.full(n_total, np.nan)
    oof_sar = np.full(n_total, np.nan)
    oof_gru = np.full(n_total, np.nan)

    for fold_i, (tr_idx, te_idx) in enumerate(tscv.split(D_eff_arr), 1):
        D_train = D_eff_arr.iloc[tr_idx]
        D_test  = D_eff_arr.iloc[te_idx]
        print(f"\n--- Klostis {fold_i}/{K_FOLDS}: train={len(D_train)} d. "
              f"({D_train.index.min().date()} -> {D_train.index.max().date()}), "
              f"test={len(D_test)} d. ({D_test.index.min().date()} -> {D_test.index.max().date()}) ---")

        t0 = time.time()
        print(f"  [1/3] XGBoost OOF...", end=" ", flush=True)
        oof_xgb[te_idx] = _oof_xgb(D_train, D_test, xgb_params)
        print(f"({time.time()-t0:.0f}s)")

        t0 = time.time()
        print(f"  [2/3] SARIMAX OOF...", end=" ", flush=True)
        oof_sar[te_idx] = _oof_sarimax(D_train, D_test, SAR_ORDER, SAR_SEASONAL_ORDER)
        print(f"({time.time()-t0:.0f}s)")

        t0 = time.time()
        print(f"  [3/3] GRU OOF...", end=" ", flush=True)
        oof_gru[te_idx] = _oof_gru(D_train, D_test, gru_params)
        print(f"({time.time()-t0:.0f}s)")

    #oof aibe - tik kur visi 3 turi prognoze
    oof_mask = ~(np.isnan(oof_xgb) | np.isnan(oof_sar) | np.isnan(oof_gru))
    oof_dates  = D_eff_arr.index[oof_mask]
    oof_y_true = D_eff_arr.loc[oof_dates, "Skambuciai"].values.astype(float)
    oof_xgb    = oof_xgb[oof_mask]
    oof_sar    = oof_sar[oof_mask]
    oof_gru    = oof_gru[oof_mask]
    print(f"\nOOF mokymo aibe: {len(oof_dates)} d. ({oof_dates.min().date()} -> {oof_dates.max().date()})")

    #kokybes patikra
    print("\n=== OOF kokybe ===")
    print(f"{'Modelis':10s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>7s}")
    for name, oof in [("XGBoost", oof_xgb), ("SARIMAX", oof_sar), ("GRU", oof_gru)]:
        print(f"{name:10s} {mae(oof_y_true, oof):8.1f} {rmse(oof_y_true, oof):8.1f} {mape(oof_y_true, oof):6.2f}%")

    #saugom cache
    cache_data = {
        "_train_end": str(TRAIN_END.date()),
        "dates":   [str(d.date()) for d in oof_dates],
        "y_true":  oof_y_true.tolist(),
        "xgb":     oof_xgb.tolist(),
        "sarimax": oof_sar.tolist(),
        "gru":     oof_gru.tolist(),
    }
    with open(OOF_CACHE, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)
    print(f"\nOOF cache issaugotas: {OOF_CACHE}")


In [ ]:
#meta-modelio ivestys: 3 baziniu prognozes + 6 kontekstiniai pozymiai

#kontekstiniai pozymiai
META_CONTEXT_COLS = ["is_holiday", "is_weekend", "dow_sin", "dow_cos", "moy_sin", "moy_cos"]
context_df = df_feat.loc[oof_dates, META_CONTEXT_COLS].astype(float)

#level-1 mokymo aibe
X_meta = np.column_stack([
    oof_xgb,
    oof_sar,
    oof_gru,
    context_df.values,
])
y_meta = oof_y_true.copy()

META_FEATURE_NAMES = ["xgb_pred", "sar_pred", "gru_pred"] + META_CONTEXT_COLS
META_BASE_NAMES    = ["xgb_pred", "sar_pred", "gru_pred"]

print(f"Level-1 mokymo aibe: X={X_meta.shape}, y={y_meta.shape}")
print(f"Pozymiai: {META_FEATURE_NAMES}")

#paskutiniu 20% chronologinis holdout palyginimui
HOLDOUT_FRAC = 0.20
n_holdout = int(len(X_meta) * HOLDOUT_FRAC)
n_train_meta = len(X_meta) - n_holdout

X_meta_tr, y_meta_tr = X_meta[:n_train_meta], y_meta[:n_train_meta]
X_meta_te, y_meta_te = X_meta[n_train_meta:], y_meta[n_train_meta:]
holdout_dates = oof_dates[n_train_meta:]

print(f"Train: {n_train_meta} d., Holdout: {n_holdout} d. ({holdout_dates.min().date()} -> {holdout_dates.max().date()})")

#koreliaciju matrica - jei tarp kintamuju stiprus rysys, gali reiketi ridge modelio
print("\n=== Baziniu prognoziu koreliacijos (OOF) ===")
corr_base = pd.DataFrame(X_meta[:, :3], columns=["XGBoost","SARIMAX","GRU"]).corr()
print(corr_base.round(3))
if corr_base.values[np.triu_indices(3, k=1)].max() > 0.95:
    print("\nAukstas korreliacija (>0.95) - Ridge L2 butina")


In [ ]:
#meta-modeliu mokymas: ridge, nnls, seklesnis xgboost
#geriausia pasirenkam pagal mase ant chronologinio holdout

from sklearn.linear_model import RidgeCV, Ridge
from scipy.optimize import nnls
import xgboost as xgb_lib

if os.path.exists(META_PARAMS):
    with open(META_PARAMS, "r", encoding="utf-8") as f:
        meta_data = json.load(f)
    meta_data.pop("_train_end", None)
    print(f"Meta-modelio parametrai pakrauti is {META_PARAMS}")
    print(f"   Geriausias: {meta_data['best_name']}")
else:
    print("Treniruojam 3 meta-modelius...\n")

    results = {}

    #1. ridge regresija
    print("[1/3] Ridge regresija (RidgeCV)...")
    alphas_grid = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    ridge_cv = RidgeCV(alphas=alphas_grid, cv=5).fit(X_meta_tr, y_meta_tr)
    pred_ridge = ridge_cv.predict(X_meta_te)
    pred_ridge = np.maximum(0.0, pred_ridge)
    metrics_ridge = {
        "MAE":  mae(y_meta_te, pred_ridge),
        "RMSE": rmse(y_meta_te, pred_ridge),
        "MAPE": mape(y_meta_te, pred_ridge),
        "MASE": mase(y_meta_te, pred_ridge, y_meta_tr),
    }
    results["Ridge"] = {
        "metrics":      metrics_ridge,
        "predictions":  pred_ridge.tolist(),
        "params": {
            "alpha":      float(ridge_cv.alpha_),
            "intercept":  float(ridge_cv.intercept_),
            "coef":       ridge_cv.coef_.tolist(),
        }
    }
    print(f"   alpha={ridge_cv.alpha_:.4f}, MAE={metrics_ridge['MAE']:.1f}, MAPE={metrics_ridge['MAPE']:.2f}%, MASE={metrics_ridge['MASE']:.3f}")
    print(f"   Koeficientai: {dict(zip(META_FEATURE_NAMES, ridge_cv.coef_.round(4)))}")

    #2. nnls (svoriai >= 0, suma = 1)
    print("\n[2/3] NNLS (Non-Negative Least Squares)...")
    X_base = X_meta_tr[:, :3]
    coef_nnls, residual = nnls(X_base, y_meta_tr)
    if coef_nnls.sum() > 0:
        coef_nnls_norm = coef_nnls / coef_nnls.sum()
    else:
        coef_nnls_norm = np.array([1/3, 1/3, 1/3])
    pred_nnls_base = X_meta_te[:, :3] @ coef_nnls_norm
    bias_corr = float(y_meta_tr.mean() - X_meta_tr[:, :3].mean(axis=0) @ coef_nnls_norm)
    pred_nnls = pred_nnls_base + bias_corr
    pred_nnls = np.maximum(0.0, pred_nnls)
    metrics_nnls = {
        "MAE":  mae(y_meta_te, pred_nnls),
        "RMSE": rmse(y_meta_te, pred_nnls),
        "MAPE": mape(y_meta_te, pred_nnls),
        "MASE": mase(y_meta_te, pred_nnls, y_meta_tr),
    }
    results["NNLS"] = {
        "metrics":     metrics_nnls,
        "predictions": pred_nnls.tolist(),
        "params": {
            "weights":     coef_nnls_norm.tolist(),
            "bias":        bias_corr,
            "weights_raw": coef_nnls.tolist(),
        }
    }
    print(f"   Svoriai: XGBoost={coef_nnls_norm[0]:.3f}, SARIMAX={coef_nnls_norm[1]:.3f}, GRU={coef_nnls_norm[2]:.3f} (suma=1.0)")
    print(f"   bias={bias_corr:.1f}, MAE={metrics_nnls['MAE']:.1f}, MAPE={metrics_nnls['MAPE']:.2f}%, MASE={metrics_nnls['MASE']:.3f}")

    #3. seklesnis xgboost
    print("\n[3/3] Seklesnis XGBoost (max_depth=3, n_est=50)...")
    meta_xgb = xgb_lib.XGBRegressor(
        objective="reg:squarederror",
        max_depth=3, n_estimators=50,
        learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, tree_method="hist",
    )
    meta_xgb.fit(X_meta_tr, y_meta_tr)
    pred_xgb_meta = meta_xgb.predict(X_meta_te)
    pred_xgb_meta = np.maximum(0.0, pred_xgb_meta)
    metrics_xgb_meta = {
        "MAE":  mae(y_meta_te, pred_xgb_meta),
        "RMSE": rmse(y_meta_te, pred_xgb_meta),
        "MAPE": mape(y_meta_te, pred_xgb_meta),
        "MASE": mase(y_meta_te, pred_xgb_meta, y_meta_tr),
    }
    booster = meta_xgb.get_booster()
    raw_bytes = booster.save_raw()
    import base64
    raw_b64 = base64.b64encode(bytes(raw_bytes)).decode("ascii")
    feat_imp = dict(zip(META_FEATURE_NAMES, meta_xgb.feature_importances_.tolist()))
    results["XGBoostMeta"] = {
        "metrics":     metrics_xgb_meta,
        "predictions": pred_xgb_meta.tolist(),
        "params": {
            "feature_importances": feat_imp,
            "model_b64":           raw_b64,
        }
    }
    print(f"   MAE={metrics_xgb_meta['MAE']:.1f}, MAPE={metrics_xgb_meta['MAPE']:.2f}%, MASE={metrics_xgb_meta['MASE']:.3f}")
    print(f"   Top svarba: {sorted(feat_imp.items(), key=lambda kv: -kv[1])[:3]}")

    #renkam geriausia
    print("\n=== Meta-modeliu palyginimas (chronologinis holdout) ===")
    print(f"{'Modelis':<14s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>7s} {'MASE':>8s}")
    for name in ["Ridge", "NNLS", "XGBoostMeta"]:
        m = results[name]["metrics"]
        print(f"{name:<14s} {m['MAE']:>8.1f} {m['RMSE']:>8.1f} {m['MAPE']:>6.2f}% {m['MASE']:>8.3f}")

    #apsauga: jei xgbmeta dominuoja viena feature - renkam nnls
    _xgb_imp = results["XGBoostMeta"]["params"].get("feature_importances", {})
    _max_imp = max(_xgb_imp.values()) if _xgb_imp else 0.0
    _xgb_dominated = _max_imp > 0.50
    _sorted = sorted(results.keys(), key=lambda k: results[k]["metrics"]["MASE"])
    _best_raw = _sorted[0]
    if _best_raw == "XGBoostMeta" and _xgb_dominated:
        print(f"[!] XGBoostMeta dominuoja viena feature ({_max_imp*100:.1f}%) - renkam NNLS.")
        best_name = "NNLS"
    elif _best_raw == "XGBoostMeta" and results["NNLS"]["metrics"]["MASE"]/results["XGBoostMeta"]["metrics"]["MASE"] < 1.05:
        print(f"[i] NNLS panasus - renkam NNLS interpretuojamumui.")
        best_name = "NNLS"
    else:
        best_name = _best_raw
    print(f"\nGeriausias meta-modelis: {best_name} (MASE={results[best_name]['metrics']['MASE']:.3f})")

    meta_data = {
        "_train_end":    str(TRAIN_END.date()),
        "best_name":     best_name,
        "feature_names": META_FEATURE_NAMES,
        "all_results":   {k: {"metrics": v["metrics"], "params": v["params"]}
                          for k, v in results.items()},
    }
    with open(META_PARAMS, "w", encoding="utf-8") as f:
        json.dump(meta_data, f, indent=2)
    print(f"\nMeta-modelio parametrai issaugoti: {META_PARAMS}")

#geriausio meta-modelio rekonstrukcija
BEST_META = meta_data["best_name"]
print(f"\nNaudojamas meta-modelis: {BEST_META}")


In [ ]:
#savaitine prognoze + zurnalas
#hibridinis nepertreniruoja baziniu - skaitomos ju savaitines prognozes is *_zurnalas.csv

def kita_savaite(paskutine: pd.Timestamp):
    start = paskutine + pd.Timedelta(days=1)
    if start > BLIND_END:
        return None
    days_until_sunday = 6 - start.dayofweek
    week_end = start + pd.Timedelta(days=days_until_sunday)
    week_end = min(week_end, BLIND_END)
    return pd.date_range(start, week_end, freq="D")

def apply_meta_model(X_input):
    if BEST_META == "Ridge":
        coef = np.array(meta_data["all_results"]["Ridge"]["params"]["coef"])
        intercept = meta_data["all_results"]["Ridge"]["params"]["intercept"]
        return X_input @ coef + intercept
    elif BEST_META == "NNLS":
        weights = np.array(meta_data["all_results"]["NNLS"]["params"]["weights"])
        bias = meta_data["all_results"]["NNLS"]["params"]["bias"]
        return X_input[:, :3] @ weights + bias
    elif BEST_META == "XGBoostMeta":
        import xgboost as xgb_lib
        import base64
        raw_b64 = meta_data["all_results"]["XGBoostMeta"]["params"]["model_b64"]
        booster = xgb_lib.Booster()
        booster.load_model(bytearray(base64.b64decode(raw_b64)))
        d_in = xgb_lib.DMatrix(X_input)
        return booster.predict(d_in)
    else:
        raise ValueError(f"Nezinomas meta: {BEST_META}")

#bootstrap CI
def bootstrap_meta_ci(X_input, n_bootstrap=BOOTSTRAP_N, alpha=ALPHA_CI):
    #naudoja ridge net jei pagrindinis xgboostmeta - kad bootstrap butu interpretuojamas
    from sklearn.linear_model import Ridge as _Ridge
    n_oof = len(X_meta_tr)
    rng = np.random.RandomState(SEED)
    boot_preds = np.zeros((n_bootstrap, len(X_input)))
    alpha_ridge = float(meta_data["all_results"]["Ridge"]["params"]["alpha"])

    for b in range(n_bootstrap):
        idx = rng.choice(n_oof, size=n_oof, replace=True)
        m_b = _Ridge(alpha=alpha_ridge).fit(X_meta_tr[idx], y_meta_tr[idx])
        boot_preds[b] = m_b.predict(X_input)

    lo = np.percentile(boot_preds, 100 * alpha / 2, axis=0)
    hi = np.percentile(boot_preds, 100 * (1 - alpha / 2), axis=0)
    return np.maximum(0.0, lo), np.maximum(0.0, hi)

#baziniu zurnalu nuskaitymas
def _read_log(path, model_name):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Truksta {model_name} zurnalo: {path}\n"
            f"Pirma paleisk {model_name} notebooka."
        )
    z = pd.read_csv(path, parse_dates=["Data"])
    z = z.set_index("Data").sort_index()
    return z

xgb_log = _read_log(XGB_LOG, "XGBoost")
sar_log = _read_log(SAR_LOG, "SARIMAX")
gru_log = _read_log(GRU_LOG, "GRU")

print(f"XGBoost zurnalas: {len(xgb_log)} d. (paskutine: {xgb_log.index.max().date()})")
print(f"SARIMAX zurnalas: {len(sar_log)} d. (paskutine: {sar_log.index.max().date()})")
print(f"GRU zurnalas:     {len(gru_log)} d. (paskutine: {gru_log.index.max().date()})")

#bendras indeksas (visi 3 turi prognoze)
common_idx = xgb_log.index.intersection(sar_log.index).intersection(gru_log.index)
print(f"\nBendras 3 zurnalu laikotarpis: {len(common_idx)} d.")
if len(common_idx) == 0:
    raise RuntimeError("Nera bendro laikotarpio. Ar visi 3 paleisti?")

#hibridinis zurnalas
STULP = ["Savaites_Nr","Prognoze","CI_Lower","CI_Upper","Faktas","Abs_Klaida","Proc_Klaida"]
if os.path.exists(LOG_PATH):
    zurnalas = pd.read_csv(LOG_PATH, index_col="Data", parse_dates=True)
    truksta = [c for c in STULP if c not in zurnalas.columns]
    for c in truksta:
        zurnalas[c] = np.nan
    zurnalas = zurnalas[STULP]
else:
    zurnalas = pd.DataFrame(columns=STULP)
    zurnalas.index = pd.DatetimeIndex([], name="Data")
print(f"Hibridinis zurnalas: {len(zurnalas)} irasu")

#trukstancios prognozes
context_2026 = sukurti_pozymius(df[["Skambuciai"]])[META_CONTEXT_COLS]

trukstancios_dt = [d for d in common_idx
                   if d not in zurnalas.index or pd.isna(zurnalas.loc[d, "Prognoze"])]

if len(trukstancios_dt) > 0:
    print(f"\nGeneruojam hibridines prognozes {len(trukstancios_dt)} dienoms...")

    rows_X = []
    rows_dates = []
    for dt in trukstancios_dt:
        if dt not in context_2026.index:
            continue
        x_row = [
            float(xgb_log.loc[dt, "Prognoze"]),
            float(sar_log.loc[dt, "Prognoze"]),
            float(gru_log.loc[dt, "Prognoze"]),
        ]
        ctx = context_2026.loc[dt].astype(float).values.tolist()
        rows_X.append(x_row + ctx)
        rows_dates.append(dt)

    if len(rows_X) > 0:
        X_input = np.array(rows_X, dtype=float)
        y_hybrid = apply_meta_model(X_input)
        y_hybrid = np.maximum(0.0, y_hybrid)
        print(f"  Bootstrap CI ({BOOTSTRAP_N} imciu)...")
        ci_lo, ci_hi = bootstrap_meta_ci(X_input, n_bootstrap=BOOTSTRAP_N)

        for i, dt in enumerate(rows_dates):
            sav_nr = savaites_nr_metuose(dt)
            zurnalas.loc[dt] = {
                "Savaites_Nr": sav_nr,
                "Prognoze":    int(round(float(y_hybrid[i]))),
                "CI_Lower":    int(round(max(0.0, float(ci_lo[i])))),
                "CI_Upper":    int(round(max(0.0, float(ci_hi[i])))),
                "Faktas":      np.nan,
                "Abs_Klaida":  np.nan,
                "Proc_Klaida": np.nan,
            }
        zurnalas = zurnalas.sort_index()
        print(f"  Sugeneruota {len(rows_dates)} prognoziu")

#uzpildom faktais
uzpildyta = 0
for dt in zurnalas.index:
    if pd.isna(zurnalas.at[dt, "Faktas"]) and dt in df_feat.index \
            and not pd.isna(df_feat.at[dt, "Skambuciai"]):
        f = float(df_feat.at[dt, "Skambuciai"])
        p = float(zurnalas.at[dt, "Prognoze"])
        zurnalas.at[dt, "Faktas"]      = f
        zurnalas.at[dt, "Abs_Klaida"]  = abs(f - p)
        zurnalas.at[dt, "Proc_Klaida"] = abs(f - p) / f * 100 if f != 0 else np.nan
        uzpildyta += 1
if uzpildyta:
    print(f"\nUzpildyta {uzpildyta} faktu")

#paskutines savaites paklaida
uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta):
    pask_nr = int(uzbaigta["Savaites_Nr"].max())
    grp = uzbaigta[uzbaigta["Savaites_Nr"] == pask_nr]
    if len(grp) >= 7:
        y_f = grp["Faktas"].values.astype(float)
        y_p = grp["Prognoze"].values.astype(float)
        print(f"\n--- Praejusi savaite {pask_nr}: {grp.index.min().date()} -> {grp.index.max().date()} ---")
        print(f"  MAE  : {mae(y_f, y_p):8.1f}")
        print(f"  RMSE : {rmse(y_f, y_p):8.1f}")
        print(f"  MAPE : {mape(y_f, y_p):8.2f}%")
        print(f"  MASE : {mase(y_f, y_p, D_eff['Skambuciai'].values):8.2f}")

zurnalas.index.name = "Data"
zurnalas.to_csv(LOG_PATH, encoding="utf-8")
print(f"\nHibridinis zurnalas issaugotas: {LOG_PATH} ({len(zurnalas)} eil.)")

#naujausios prognozes
neuzpildytos = zurnalas[zurnalas["Faktas"].isna()]
if len(neuzpildytos):
    print(f"\nNaujausios hibridines prognozes (laukia faktu):")
    for dt in neuzpildytos.index[-7:]:
        diena = ["Pir","Ant","Tre","Ket","Pen","Ses","Sek"][dt.dayofweek]
        p = int(zurnalas.at[dt, "Prognoze"])
        lo = int(zurnalas.at[dt, "CI_Lower"])
        hi = int(zurnalas.at[dt, "CI_Upper"])
        print(f"  {dt.date()} ({diena}): {p:5d}   [90% CI: {lo:5d} - {hi:5d}]")


In [ ]:
#grafikai, metrikos, svoriu dinamika

import matplotlib.dates as mdates

uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta) == 0:
    print("Dar nera 2026 faktu - grafika ir metrikos bus po pirmuju savaiciu.")
else:
    #pagrindiniai metrikai
    y_f = uzbaigta["Faktas"].values.astype(float)
    y_p = uzbaigta["Prognoze"].values.astype(float)
    lo  = uzbaigta["CI_Lower"].values.astype(float)
    hi  = uzbaigta["CI_Upper"].values.astype(float)
    y_range = y_f.max() - y_f.min()

    print("=== Hibridinio modelio metrikos ===")
    print(f"  Imtis:    {len(uzbaigta)} d.")
    print(f"  MAE:      {mae(y_f, y_p):8.1f}")
    print(f"  RMSE:     {rmse(y_f, y_p):8.1f}")
    print(f"  MAPE:     {mape(y_f, y_p):8.2f}%")
    print(f"  sMAPE:    {smape(y_f, y_p):8.2f}%")
    print(f"  MASE:     {mase(y_f, y_p, D_eff['Skambuciai'].values):8.2f}")
    print(f"  PICP:     {picp(y_f, lo, hi):8.2f}%  (tikslas 90.0)")
    print(f"  PINAW:    {pinaw(lo, hi, y_range):8.4f}")
    print(f"  Winkler:  {winkler(y_f, lo, hi):8.1f}")

    #grafikai
    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=False)

    ax1 = axes[0]
    ax1.plot(uzbaigta.index, uzbaigta["Faktas"], "k-", lw=2.0, label="Faktas", marker="s", ms=4)
    ax1.plot(uzbaigta.index, uzbaigta["Prognoze"], color="#2f855a", lw=2.0,
             label=f"Hibridinis ({BEST_META})", marker="o", ms=4)
    ax1.fill_between(uzbaigta.index, uzbaigta["CI_Lower"], uzbaigta["CI_Upper"],
                     alpha=0.15, color="#2f855a", label="90% CI")
    common_filled = [d for d in uzbaigta.index if d in xgb_log.index]
    if common_filled:
        ax1.plot(common_filled, xgb_log.loc[common_filled, "Prognoze"],
                 "--", color="#1f77b4", alpha=0.6, lw=1.0, label="XGBoost")
        ax1.plot(common_filled, sar_log.loc[common_filled, "Prognoze"],
                 "--", color="#d62728", alpha=0.6, lw=1.0, label="SARIMAX")
        ax1.plot(common_filled, gru_log.loc[common_filled, "Prognoze"],
                 "--", color="#7c4dff", alpha=0.6, lw=1.0, label="GRU")

    ax1.set_title("2026 Q1: hibridinis vs baziniai vs faktas")
    ax1.set_xlabel("Data"); ax1.set_ylabel("Skambuciu skaicius")
    ax1.legend(loc="upper right", ncol=3, fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

    #kumuliatyvine suma
    ax2 = axes[1]
    ax2.plot(uzbaigta.index, uzbaigta["Faktas"].cumsum(), "k-", lw=2.0, label="Faktas (kumul.)")
    ax2.plot(uzbaigta.index, uzbaigta["Prognoze"].cumsum(), color="#2f855a",
             lw=2.0, label="Hibridinis (kumul.)")
    if common_filled:
        ax2.plot(common_filled, xgb_log.loc[common_filled, "Prognoze"].cumsum(),
                 "--", color="#1f77b4", alpha=0.6, label="XGBoost (kumul.)")
        ax2.plot(common_filled, sar_log.loc[common_filled, "Prognoze"].cumsum(),
                 "--", color="#d62728", alpha=0.6, label="SARIMAX (kumul.)")
        ax2.plot(common_filled, gru_log.loc[common_filled, "Prognoze"].cumsum(),
                 "--", color="#7c4dff", alpha=0.6, label="GRU (kumul.)")
    ax2.set_title("Kumuliatyvine suma")
    ax2.set_xlabel("Data"); ax2.set_ylabel("Suminis skaicius")
    ax2.legend(loc="upper left", ncol=2, fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

#meta-modelio svoriu vizualizacija
fig, ax = plt.subplots(figsize=(10, 5))
if BEST_META == "Ridge":
    coef = np.array(meta_data["all_results"]["Ridge"]["params"]["coef"])
    colors = ["#1f77b4","#d62728","#7c4dff"] + ["#888"] * (len(META_FEATURE_NAMES) - 3)
    bars = ax.bar(META_FEATURE_NAMES, coef, color=colors)
    ax.set_title(f"Ridge meta-modelio koeficientai (alpha={meta_data['all_results']['Ridge']['params']['alpha']:.4f})")
    ax.axhline(0, color="black", lw=0.5)
    for b, c in zip(bars, coef):
        ax.text(b.get_x() + b.get_width()/2, c, f"{c:.3f}", ha="center",
                va="bottom" if c >= 0 else "top", fontsize=9)
elif BEST_META == "NNLS":
    weights = np.array(meta_data["all_results"]["NNLS"]["params"]["weights"])
    colors = ["#1f77b4","#d62728","#7c4dff"]
    ax.bar(["XGBoost","SARIMAX","GRU"], weights, color=colors)
    ax.set_title(f"NNLS svoriai (suma=1.0, bias={meta_data['all_results']['NNLS']['params']['bias']:.1f})")
    for i, w in enumerate(weights):
        ax.text(i, w, f"{w:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
elif BEST_META == "XGBoostMeta":
    feat_imp = meta_data["all_results"]["XGBoostMeta"]["params"]["feature_importances"]
    items = sorted(feat_imp.items(), key=lambda kv: -kv[1])
    ax.bar([k for k,v in items], [v for k,v in items], color="#1f77b4")
    ax.set_title("XGBoost meta-modelio pozymiu svarba")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

ax.set_ylabel("Koef. / svoris" if BEST_META != "XGBoostMeta" else "Svarba")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

#3 meta-modeliu palyginimas
print("\n=== 3 meta-modeliu palyginimas ===")
print(f"{'Modelis':<14s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>7s} {'MASE':>8s} {'Pasirinkta':>11s}")
for name in ["Ridge", "NNLS", "XGBoostMeta"]:
    m = meta_data["all_results"][name]["metrics"]
    chosen = "TAIP" if name == BEST_META else ""
    print(f"{name:<14s} {m['MAE']:>8.1f} {m['RMSE']:>8.1f} {m['MAPE']:>6.2f}% {m['MASE']:>8.3f} {chosen:>11s}")


In [ ]:
#dm-hln + holm-bonferroni 
#hibridinis statistiskai lenkia visus 3 bazinius

from scipy.stats import t as student_t

def dm_hln(e1, e2, h=1, loss="MSE"):
    e1 = np.asarray(e1, dtype=float)
    e2 = np.asarray(e2, dtype=float)
    if loss == "MSE":
        d = e1**2 - e2**2
    elif loss == "MAE":
        d = np.abs(e1) - np.abs(e2)
    else:
        raise ValueError(f"Nezinoma: {loss}")

    n = len(d)
    d_mean = float(np.mean(d))
    gamma_0 = float(np.var(d, ddof=0))
    var_d = gamma_0
    for k in range(1, h):
        gamma_k = float(np.mean((d[:-k] - d_mean) * (d[k:] - d_mean)))
        var_d += 2 * gamma_k
    if var_d <= 0:
        return np.nan, np.nan, n - 1

    dm = d_mean / np.sqrt(var_d / n)
    correction = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    dm_hln_stat = dm * correction
    df = n - 1
    p_value = 2 * (1 - student_t.cdf(abs(dm_hln_stat), df))
    return float(dm_hln_stat), float(p_value), df

uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta) < 7:
    print("Per maza imtis (reikia >=7 d.).")
else:
    common_filled = [d for d in uzbaigta.index
                     if d in xgb_log.index and d in sar_log.index and d in gru_log.index
                     and not pd.isna(xgb_log.loc[d, "Faktas"])
                     and not pd.isna(sar_log.loc[d, "Faktas"])
                     and not pd.isna(gru_log.loc[d, "Faktas"])]

    if len(common_filled) < 7:
        print(f"Per mazai bendru dienu ({len(common_filled)} d.)")
    else:
        y_f = uzbaigta.loc[common_filled, "Faktas"].values.astype(float)
        e_hyb = y_f - uzbaigta.loc[common_filled, "Prognoze"].values.astype(float)
        e_xgb = y_f - xgb_log.loc[common_filled, "Prognoze"].values.astype(float)
        e_sar = y_f - sar_log.loc[common_filled, "Prognoze"].values.astype(float)
        e_gru = y_f - gru_log.loc[common_filled, "Prognoze"].values.astype(float)

        print(f"=== DM-HLN: hibridinis vs baziniai (n={len(common_filled)}) ===")
        print("e_hibridinis ir e_X yra lygiavertes paklaidos")
        print("MSE pagrindu, abi puses (h=1)\n")

        results = []
        for name, e_other in [("XGBoost", e_xgb), ("SARIMAX", e_sar), ("GRU", e_gru)]:
            dm_stat, p_val, df = dm_hln(e_hyb, e_other, h=1, loss="MSE")
            mse_hyb = float(np.mean(e_hyb**2))
            mse_oth = float(np.mean(e_other**2))
            laimi = "Hibridinis" if mse_hyb < mse_oth else name
            results.append({"vs": name, "dm": dm_stat, "p": p_val, "laimi": laimi,
                            "mse_hyb": mse_hyb, "mse_oth": mse_oth})

        #holm-bonferroni
        sorted_p = sorted([(r["vs"], r["p"]) for r in results], key=lambda x: x[1])
        m = len(sorted_p)
        alpha_holm = {}
        for i, (name, p) in enumerate(sorted_p):
            alpha_holm[name] = 0.05 / (m - i)

        print(f"{'vs Modelis':12s} {'DM-HLN':>10s} {'p-value':>10s} {'Holm alpha':>11s} {'Reiksminga?':>13s} {'Laimi':>14s}")
        for r in results:
            ah = alpha_holm[r["vs"]]
            sig = "TAIP" if r["p"] < ah else "ne"
            print(f"{r['vs']:12s} {r['dm']:>10.3f} {r['p']:>10.4f} {ah:>11.4f} {sig:>13s} {r['laimi']:>14s}")

        #verdiktas
        wins = sum(1 for r in results if r["p"] < alpha_holm[r["vs"]] and r["laimi"] == "Hibridinis")
        print(f"\nHibridinis statistiskai lenkia {wins}/3 baziniu modeliu.")
        if wins == 3:
            print("Patvirtinta: hibridinis lenkia visus.")
        elif wins >= 1:
            print(f"Dalinai patvirtinta: lenkia {wins} is 3.")
        else:
            print("Nepatvirtinta esamoje imtyje.")


In [ ]:
#kontekstiniu efektu analize 
#modeliu svoriai priklauso nuo kontekstu

if BEST_META == "Ridge":
    coef = np.array(meta_data["all_results"]["Ridge"]["params"]["coef"])
    intercept = meta_data["all_results"]["Ridge"]["params"]["intercept"]
    print("=== Ridge meta-modelio koeficientai ===\n")
    print(f"Intercept (b0): {intercept:+.2f}")
    print(f"Bazinis svoris XGBoost (b1):  {coef[0]:+.4f}")
    print(f"Bazinis svoris SARIMAX (b2): {coef[1]:+.4f}")
    print(f"Bazinis svoris GRU (b3):     {coef[2]:+.4f}")
    print(f"\nKontekstiniai koeficientai:")
    for i, name in enumerate(META_CONTEXT_COLS):
        idx = i + 3
        v = coef[idx]
        interp = ""
        if name == "is_holiday":
            interp = "(sventes itaka)"
        elif name == "is_weekend":
            interp = "(savaitgalio itaka)"
        elif name.startswith("dow"):
            interp = "(savaites dienos cikline)"
        elif name.startswith("moy"):
            interp = "(menesio cikline)"
        print(f"  gamma_{name:<14s}: {v:+10.4f}  {interp}")

    #ar reiksmingi (paprasta praktine taisykle)
    std_y = float(np.std(y_meta))
    significant_ctx = [(name, coef[i+3]) for i, name in enumerate(META_CONTEXT_COLS)
                       if abs(coef[i+3]) > 0.5 * std_y / 10]
    if significant_ctx:
        print(f"\nReiksmingi kontekstiniai (|gamma| > 0.05*std(y)):")
        for name, v in significant_ctx:
            print(f"  {name}: {v:+.4f}")
        print("\nDalinai patvirtinta: kontekstas itakoja prognoze.")
    else:
        print("\nKontekstiniai nedideli - silpnai patvirtinta.")

elif BEST_META == "NNLS":
    print("=== NNLS svoriai ===\n")
    weights = np.array(meta_data["all_results"]["NNLS"]["params"]["weights"])
    bias = meta_data["all_results"]["NNLS"]["params"]["bias"]
    print(f"  Svoriai (suma=1.0):")
    for name, w in zip(["XGBoost","SARIMAX","GRU"], weights):
        print(f"    {name:8s}: {w:.4f} ({w*100:.1f}%)")
    print(f"  Bias korekcija: {bias:+.2f}")
    print("\nNNLS yra kontekstui-nepriklausomas - kontekstinius efektus matom tik per ridge.")

elif BEST_META == "XGBoostMeta":
    print("=== XGBoost meta-modelio SHAP analize ===\n")
    try:
        import shap
        import xgboost as xgb_lib
        import base64
        raw_b64 = meta_data["all_results"]["XGBoostMeta"]["params"]["model_b64"]
        booster = xgb_lib.Booster()
        booster.load_model(bytearray(base64.b64decode(raw_b64)))
        explainer = shap.TreeExplainer(booster)
        shap_vals = explainer.shap_values(X_meta_te[:50])
        mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        order = np.argsort(-mean_abs_shap)
        print("Vidutine absoliucia SHAP svarba:")
        for i in order:
            print(f"  {META_FEATURE_NAMES[i]:<18s}: {mean_abs_shap[i]:8.2f}")
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh([META_FEATURE_NAMES[i] for i in order],
                [mean_abs_shap[i] for i in order], color="#1f77b4")
        ax.set_xlabel("Vidutine absoliucia SHAP svarba")
        ax.set_title("XGBoost meta-modelio pozymiu svarba")
        ax.invert_yaxis()
        plt.tight_layout(); plt.show()
    except ImportError:
        print("shap nepasiekiamas. Naudojam feature importance:")
        feat_imp = meta_data["all_results"]["XGBoostMeta"]["params"]["feature_importances"]
        for k, v in sorted(feat_imp.items(), key=lambda kv: -kv[1]):
            print(f"  {k:<18s}: {v:.4f}")


In [ ]:
#prognoziu intervalu kokybe - 4 modeliai

uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta) < 7:
    print("Per maza imtis CI palyginimui.")
else:
    common_filled = [d for d in uzbaigta.index
                     if d in xgb_log.index and not pd.isna(xgb_log.loc[d, "Faktas"])]
    if not common_filled:
        print("Nera bendru dienu.")
    else:
        y_f_all = uzbaigta.loc[common_filled, "Faktas"].values.astype(float)
        y_range = y_f_all.max() - y_f_all.min()

        results = {}
        results["Hibridinis"] = {
            "PICP":    picp(y_f_all,
                            uzbaigta.loc[common_filled, "CI_Lower"].values.astype(float),
                            uzbaigta.loc[common_filled, "CI_Upper"].values.astype(float)),
            "PINAW":   pinaw(uzbaigta.loc[common_filled, "CI_Lower"].values.astype(float),
                             uzbaigta.loc[common_filled, "CI_Upper"].values.astype(float),
                             y_range),
            "Winkler": winkler(y_f_all,
                               uzbaigta.loc[common_filled, "CI_Lower"].values.astype(float),
                               uzbaigta.loc[common_filled, "CI_Upper"].values.astype(float)),
        }
        for name, log in [("XGBoost", xgb_log), ("SARIMAX", sar_log), ("GRU", gru_log)]:
            results[name] = {
                "PICP":    picp(y_f_all,
                                log.loc[common_filled, "CI_Lower"].values.astype(float),
                                log.loc[common_filled, "CI_Upper"].values.astype(float)),
                "PINAW":   pinaw(log.loc[common_filled, "CI_Lower"].values.astype(float),
                                 log.loc[common_filled, "CI_Upper"].values.astype(float),
                                 y_range),
                "Winkler": winkler(y_f_all,
                                   log.loc[common_filled, "CI_Lower"].values.astype(float),
                                   log.loc[common_filled, "CI_Upper"].values.astype(float)),
            }

        print(f"=== CI kokybe ({len(common_filled)} d.) ===")
        print(f"  Tikslinis PICP: 90.0%")
        print(f"\n{'Modelis':<14s} {'PICP':>8s} {'PINAW':>8s} {'Winkler':>10s}")
        for name, r in results.items():
            print(f"{name:<14s} {r['PICP']:>7.2f}% {r['PINAW']:>8.4f} {r['Winkler']:>10.1f}")

        best_picp = min(results.items(), key=lambda kv: abs(kv[1]["PICP"] - 90.0))[0]
        best_winkler = min(results.items(), key=lambda kv: kv[1]["Winkler"])[0]
        print(f"\nGeriausia PICP kalibracija (~90%): {best_picp}")
        print(f"Geriausias Winkler:                  {best_winkler}")


In [ ]:
#priedas a: train_test_split alternatyvus vertinimas
#pagrindinis vertinimas - chronologinis (P5 holdout + 2026 blind)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV as _RidgeCV

print("--- Priedas A: random train_test_split (80/20) ---")

X_trA, X_teA, y_trA, y_teA, idx_trA, idx_teA = train_test_split(
    X_meta, y_meta, oof_dates,
    test_size=0.20, random_state=SEED, shuffle=True,
)
idx_trA = pd.DatetimeIndex(idx_trA)
idx_teA = pd.DatetimeIndex(idx_teA)
print(f"Train: {len(X_trA)} d., Test: {len(X_teA)} d.")
print("\nTestu pasiskirstymas pagal metus:")
per_year = pd.Series(idx_teA.year).value_counts().sort_index()
for yr, cnt in per_year.items():
    print(f"  {yr}: {cnt} d.")

#ridge ant random train
alpha_used = float(meta_data["all_results"]["Ridge"]["params"]["alpha"])
ridge_A = _RidgeCV(alphas=[alpha_used*0.1, alpha_used, alpha_used*10],
                   cv=3).fit(X_trA, y_trA)

y_pred_A = np.maximum(0.0, ridge_A.predict(X_teA))

print(f"\n=== Priedas A - Ridge rezultatai ===")
print(f"  alpha:      {ridge_A.alpha_:.4f}")
print(f"  MAE:        {mae(y_teA, y_pred_A):8.2f}")
print(f"  RMSE:       {rmse(y_teA, y_pred_A):8.2f}")
print(f"  MAPE:       {mape(y_teA, y_pred_A):8.2f}%")
print(f"  sMAPE:      {smape(y_teA, y_pred_A):8.2f}%")
ss_res = float(np.sum((y_teA - y_pred_A) ** 2))
ss_tot = float(np.sum((y_teA - y_teA.mean()) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
print(f"  R^2:        {r2:8.4f}")

#metrikos pagal metus
print("\nPagal metus:")
print(f"  {'Metai':<6} {'n':>4} {'MAE':>8} {'MAPE':>8}")
for yr in sorted(set(idx_teA.year)):
    mask = (idx_teA.year == yr)
    if mask.sum() == 0: continue
    y_yr = y_teA[mask]
    p_yr = y_pred_A[mask]
    print(f"  {yr:<6} {mask.sum():>4} {mae(y_yr, p_yr):>8.2f} {mape(y_yr, p_yr):>8.2f}%")

#galutine isvada
print("\n" + "="*70)
print(" HIBRIDINIO MODELIO TYRIMAS BAIGTAS")
print("="*70)
print(f" Pasirinktas meta-modelis:    {BEST_META}")
print(f" OOF dydis:                    {len(oof_dates)} d.")
print(f" 2026 blind dienu su faktais: {len(uzbaigta) if 'uzbaigta' in dir() else 0}")
print(f" Failai:")
print(f"   - {LOG_PATH}")
print(f"   - {OOF_CACHE}")
print(f"   - {META_PARAMS}")
print("="*70)
